# Kotlin Patterns for Android

Read compact Kotlin patterns used to configure objects and pass actions around an app.

You already know classes, nullable values, collections, and lambdas. This lesson connects those ideas. Practice callbacks, extensions, `let`, and `apply` closely. Treat `object`, `companion object`, and `lateinit` as recognition patterns that you will revisit when building Android apps.

## Learning Goals

- Describe a function type and pass a callback that another function invokes.
- Write a small extension and choose between `let` for a result and `apply` for object configuration.
- Recognize named objects, companion objects, and the initialization requirement of `lateinit` properties.

## Why This Matters

An app needs to describe both data and actions. A draft holds a title; a confirmation action decides what happens when the user accepts that draft. Passing an action lets the receiving function use behavior supplied by its caller.

Kotlin also has compact ways to prepare an object or use an optional value. These patterns appear often in Android code. Understanding what each block receives, returns, and changes is more useful than memorizing short syntax. Our small simulations let you inspect those rules before working with real screens and events.

## Check Your Starting Point

Lesson 4 used `val makeLabel = { title: String -> title.uppercase() }`. Does that declaration print or convert any title by itself? Write a call that produces uppercase text for `Travel`. Explain the difference between defining the lambda and invoking it.

In [ ]:
Your response:
Write your explanation here.

<details>
<summary>Show answer</summary>

The declaration creates a function value. Its body has not processed a title yet. `makeLabel("Travel")` invokes it and returns `TRAVEL`; `println(makeLabel("Travel"))` also prints that result. We will now pass a function value into another function.

</details>

## Video Demonstration

Follow a draft through configuration, optional-title formatting, and a simulated confirmation callback. The example runs as an ordinary Kotlin script.

<video controls preload="metadata" width="800" aria-label="Kotlin Patterns for Android demonstration">
<source src="media/05_kotlin_patterns_for_android/lesson.mp4" type="video/mp4">
<track kind="captions" src="media/05_kotlin_patterns_for_android/captions.vtt" srclang="en" label="English">
Your browser does not support embedded video.
</video>

[Read the video transcript and visual description](media/05_kotlin_patterns_for_android/transcript.md).

## Concept

### Pass an Action with a Function Type

A **function type** describes the inputs and result of a function value. `() -> Unit` means no parameters and no meaningful result. `Unit` serves a similar purpose to `void` in Java or C++; it is not a missing value or null.

```kotlin
val action: () -> Unit = { println("Action body") }
```

The variable holds an action. `action()` invokes it. A **callback** is a function value passed to another part of a program so that part can invoke it. A parameter can use the same function type:

```kotlin
fun runAction(action: () -> Unit) {
    action()
}
```

The parameter name receives the supplied action. The parentheses inside the body actually invoke it. The receiving function determines whether and when it runs.

In [ ]:
fun runAction(action: () -> Unit) {
    println("Before action")
    action()
    println("After action")
}
val savedAction: () -> Unit = { println("Action body") }
println("Action created")
runAction(savedAction)

This prints `Action created`, `Before action`, `Action body`, and `After action`, in that order. Creating the lambda does not print its body. Here the callback runs immediately, during `runAction`.

Because the last parameter is a function, the trailing-lambda syntax from Lesson 4 also works: `runAction { println("Another action") }`. A real app may keep a callback until a later user event, but a callback is not automatically asynchronous. **Asynchronous** work can finish separately from the current sequence. This example does not schedule work or implement an Android event system.

### Add Convenient Syntax with an Extension

An **extension function** is a function you can call with member-style syntax on a type without modifying that type's class. The type before the dot is its **receiver type**. The receiver is the value on which the function is called.

```kotlin
fun String.asBadge(): String = "[$this]"
```

Inside this extension, **`this`** refers to the receiver string. `"Travel".asBadge()` therefore returns `[Travel]`. The extension creates new text; it does not change the original String class or gain access to its private members.

In [ ]:
fun String.asBadge(): String = "[$this]"
val badgeTitle = "Travel"
println(badgeTitle.asBadge())
println(badgeTitle)

The output is `[Travel]` followed by `Travel`. The receiver supplies the input, and the extension returns another string.

### Use let to Compute a Result

A **scope function** runs a block with an object available in a convenient form. `let` and `apply` are library functions, not language keywords. They differ in how the block refers to the object and what the whole call returns.

`let` passes its receiver as the block's argument, usually named `it`. The block's final expression becomes the call's result. For `"Travel".let { it.length }`, the result is the integer `6`.

With `?.let`, the safe call runs the block only when the receiver is non-null. You can follow the result with `?:` to supply a fallback. The guard checks for null, not for empty text.

In [ ]:
val selectedLabel: String? = "Travel"
val labelLength = selectedLabel?.let { it.length } ?: 0
println(labelLength)
val noLabel: String? = null
println(noLabel?.let { it.length } ?: 0)
val emptyLabel: String? = ""
emptyLabel?.let { println("Selected text: [$it]") }

This prints `6`, `0`, and `Selected text: []`. The first block computes a length. The null receiver skips its block and uses zero. The empty string is present, so its block runs.

A `let` block can also perform an action such as printing. If its final expression is `println(...)`, that expression supplies `Unit`; it does not return the printed text. To create a display string for later use, make the string expression the block's result.

### Use apply to Configure the Same Object

`apply` makes the receiver available as `this` and returns that same receiver after the block. Within the block, you can often omit `this.` when naming its properties.

```kotlin
val configured = settings.apply {
    this.title = "Travel"
    enabled = true
}
```

Here `settings` must already be an object with mutable `title` and `enabled` properties. `apply` does not create a copy. In `PreviewSettings().apply { ... }`, the constructor creates the object, and `apply` then configures it.

In [ ]:
class PreviewSettings(var title: String = "", var enabled: Boolean = false)
val previewSettings = PreviewSettings()
val configuredSettings = previewSettings.apply {
    this.title = "Travel"
    enabled = true
}
println(previewSettings.title)
println(configuredSettings.enabled)

This prints `Travel` and `true`. Both names refer to the configured object. The original reference sees the changed title because the block changed that object. This differs from the `copy` operation on a data class, which creates another instance.

| Function | Object inside the block | Result of the call | Common purpose |
| --- | --- | --- | --- |
| `let` | Argument, often `it` | Final block expression | Compute a value or use an optional value |
| `apply` | Receiver, `this` | Same receiver object | Configure an object |

Use a plain variable and separate statements when they read more clearly. Nesting several scope functions can make it hard to tell which object `it` or `this` means.

### Recognize a Named Object

An **object declaration** creates a named shared object. The `object` keyword replaces the need to declare a class and separately construct one instance for this purpose. Access its members through the object name.

In [ ]:
object AppLabels {
    val fallback = "Untitled"
    fun heading(title: String): String = "Note: $title"
}
println(AppLabels.fallback)
println(AppLabels.heading("Travel"))

The results are `Untitled` and `Note: Travel`. There is no `AppLabels()` constructor call. The declaration provides one shared object. A named object can be useful for a small shared helper, but putting mutable app data in a shared object requires care because many callers can affect the same state.

### Recognize a Companion Object

A **companion object** is an object declared inside a class with `companion object`. Its members can be accessed through the class name. It is associated with the class rather than created separately for every class instance.

A small factory function can provide a named way to create a default instance. A **factory function** is simply a function that creates and returns an object.

In [ ]:
class DraftFactory(val title: String) {
    companion object {
        fun blank(): DraftFactory = DraftFactory("Untitled")
    }
}
val blankDraft = DraftFactory.blank()
println(blankDraft.title)

This prints `Untitled`. `DraftFactory.blank()` calls the companion member through the class name. You do not need an existing `DraftFactory` instance to call it. The function then invokes the constructor and returns a new instance.

The call resembles a Java static factory call, but a Kotlin companion is an object with members. You do not need Java interoperability details for this lesson.

### Recognize lateinit and Its Ordering Rule

The **`lateinit` modifier** allows an eligible non-null mutable property to receive its first value after construction. It does not supply a default value. You must assign the property before reading it.

Use it with an eligible `var` property, such as a non-null `String`. It cannot be used with `val`, a nullable property, or a primitive type such as `Int`. Prefer an ordinary initializer or constructor parameter when the value is already available.

In [ ]:
class PreviewSession {
    lateinit var title: String
}
val previewSession = PreviewSession()
previewSession.title = "Travel"
println(previewSession.title)

This prints `Travel` because assignment happens first. The following is **intentionally incorrect** and is shown as text so it will not interrupt the notebook:

```kotlin
val unfinishedSession = PreviewSession()
println(unfinishedSession.title) // Incorrect: read before assignment.
unfinishedSession.title = "Travel"
```

The first read throws `UninitializedPropertyAccessException`, a runtime error reporting an uninitialized property. Normal execution stops before the later assignment. `lateinit` does not mean an empty string or null, and adding `!!` would not initialize the property. The repair is to establish the required value before any read.

<details class="animation-panel" open>
<summary>Trace a callback invocation — show or hide animation</summary>
<p><img src="media/05_kotlin_patterns_for_android/callback.gif" alt="An action is created and passed to simulateClick. The function invokes onClick, which prints Saved: Lab ideas once, then returns." width="960" style="max-width:100%;height:auto;"></p>
</details>

The diagram slows down an ordinary call so you can see who invokes the action. In the actual program, the callback runs immediately inside simulateClick; it is not a delayed user event. The 10-second animation loops; closing its panel hides the motion.

[View the labeled still diagram](media/05_kotlin_patterns_for_android/callback_still.png).

## Worked Example

### Configure and Confirm a Note Draft

1. `NoteDraft` holds a mutable title with an empty default.
2. `String.asNoteHeading` formats its receiver as a heading without changing it.
3. `simulateClick` accepts an action and invokes it once inside its body.
4. The constructor creates a draft. `apply` sets its title and returns that same draft.
5. `selectedTitle?.let` formats and prints the title only when the selection is non-null.
6. The trailing lambda passed to `simulateClick` prints a confirmation when `onClick()` invokes it.

The `let` block and callback are separate. Skipping the optional-title block would not prevent the later function call.

In [ ]:
class NoteDraft(var title: String = "")
fun String.asNoteHeading(): String = "Note: $this"
fun simulateClick(onClick: () -> Unit) {
    onClick()
}
val draft = NoteDraft().apply { title = "Lab ideas" }
val selectedTitle: String? = draft.title
selectedTitle?.let { println(it.asNoteHeading()) }
simulateClick { println("Saved: ${draft.title}") }

The output is:

```text
Note: Lab ideas
Saved: Lab ideas
```

The first line comes from the present title's `let` block. The second comes from the action invoked by `simulateClick`. This helper invokes the action immediately, during the ordinary function call. It does not create an Android button or wait for a screen event.

`Saved` is only printed confirmation text. This program does not write a file, contact a server, or preserve data after it ends.

## Guided Practice

### Trace present, missing, and empty titles

Before running the next cell, predict all three output lines. Identify which calls enter the `let` block and which call uses the fallback. Explain why an empty string does not take the same path as null. The square brackets are visible markers around the resulting title.

In [ ]:
Your prediction:
Three output lines:
Calls that enter let:
Call that uses the fallback:
Why the empty string takes its path:


In [ ]:
fun practiceBadgeText(title: String?): String =
    title?.let { "[$it]" } ?: "Missing"
println(practiceBadgeText("Trip"))
println(practiceBadgeText(null))
println(practiceBadgeText(""))

<details>
<summary>Show answer</summary>

```text
[Trip]
Missing
[]
```

Both `"Trip"` and `""` are non-null, so each enters `let`. Its last expression creates the bracketed string and supplies the result. Null skips `let`, leaving null for Elvis to replace with `Missing`. The empty string creates `[]`; checking for a missing value does not check whether text is empty.

</details>

### Complete an extension and a result-producing block

Write `fun String.asReminderLabel(): String` so it returns `Reminder: ` followed by its receiver string. Use `this` in the extension body. Then write `fun practicePreview(title: String?): String`: use `?.let` to call the extension on `it`, and use Elvis to return `No preview` for null. The functions should return text; print outside them.

Test with `"Read"` and null. Expect:

```text
Reminder: Read
No preview
```

Then change only the fallback to `Nothing selected` and run both tests again. The first line should stay the same; the second should become `Nothing selected`.

In [ ]:
// TODO: Define the String extension and the nullable preview function.
// Print tests for Read and null, then modify only the fallback.


<details>
<summary>Show answer</summary>

```kotlin
fun String.asReminderLabel(): String = "Reminder: $this"
fun practicePreview(title: String?): String =
    title?.let { it.asReminderLabel() } ?: "No preview"
println(practicePreview("Read"))
println(practicePreview(null))
```

Inside the extension, `this` is the receiver string. Inside `let`, `it` is the non-null string passed to that block. The extension result is the block's last expression, so `let` returns that display text. Placing only a `println` in the block would perform printing instead of returning the required label string.

For the modification, change `"No preview"` to `"Nothing selected"`. Only the null input reaches that fallback; the present title still produces `Reminder: Read`.

</details>

### Repair initialization order

The following intentionally faulty snippet compiles but fails when it reads the title:

```kotlin
class PracticeSession {
    lateinit var title: String
}
val practiceSession = PracticeSession()
println(practiceSession.title)
practiceSession.title = "Read"
```

Explain why it fails. Rewrite it so the existing assignment occurs before the first read and the program prints `Read`. Keep the non-null `lateinit var` property. Explain why assigning it later in the cell cannot repair an earlier failed read.

In [ ]:
Your diagnosis:
Why the read fails:
Required order of operations:
Why the later assignment cannot repair the earlier read:


In [ ]:
// TODO: Rewrite the snippet with initialization before the first read.


<details>
<summary>Show answer</summary>

```kotlin
class PracticeSession {
    lateinit var title: String
}
val practiceSession = PracticeSession()
practiceSession.title = "Read"
println(practiceSession.title)
```

`lateinit` postpones initialization; it does not supply an empty string or null. Reading an uninitialized property throws an `UninitializedPropertyAccessException`. The failing read stops normal execution before the later assignment is reached. Assigning `"Read"` first satisfies the initialization requirement.

This is an eligible non-null `String` `var` property. Adding `!!` would not initialize it. When the value is already available during construction, a constructor value or ordinary initializer is usually simpler; this exercise isolates the ordering rule.

</details>

## Independent Practice

### Configure a draft and confirm it

Build a program from these requirements. This is a local simulation: the callback runs during the function call, and no file or server save occurs.

- Define `ConfirmationDraft` with mutable `title: String = ""` and `ready: Boolean = false` properties.
- Define the String extension `asPreviewLine()` to return `Preview: [` followed by the receiver and `]`.
- Define `confirmDraft(onConfirmed: () -> Unit)` so its body invokes the callback once.
- Create `originalConfirmationDraft` with title `Unconfigured`. Use `apply` on that existing object to set title to `Pack bag` and ready to true. Keep the returned object as `configuredConfirmationDraft`.
- Set `selectedReminderTitle: String?` to the configured title. Use `?.let` to print its preview line only when it is non-null.
- Start `confirmationCount` at zero. Pass a callback to `confirmDraft` that increments the count and prints `Confirmed: ` followed by the configured draft title. Call `confirmDraft` once, outside the `let` block.
- After the call, print the original draft title, the configured ready flag, and the confirmation count, with the labels below.

For the present-title case, expect:

```text
Preview: [Pack bag]
Confirmed: Pack bag
Original title: Pack bag
Ready: true
Confirmations: 1
```

Test the same program with two changes to the selected-title input only:

1. Set `selectedReminderTitle` to null. The preview line disappears; all four remaining lines stay the same.
2. Set it to `""`. The first line becomes `Preview: []`; all four remaining lines stay the same.

Do not change the configured draft title, move the callback into `let`, or manually print a confirmation outside the callback to force these results.

In [ ]:
// TODO: Define the draft, extension, and callback function, then implement the steps.
// Test a present, null, and empty selected title by changing only that input.


<details>
<summary>Show answer</summary>

```kotlin
class ConfirmationDraft(var title: String = "", var ready: Boolean = false)
fun String.asPreviewLine(): String = "Preview: [$this]"
fun confirmDraft(onConfirmed: () -> Unit) {
    onConfirmed()
}
val originalConfirmationDraft = ConfirmationDraft("Unconfigured")
val configuredConfirmationDraft = originalConfirmationDraft.apply {
    title = "Pack bag"
    ready = true
}
val selectedReminderTitle: String? = configuredConfirmationDraft.title
selectedReminderTitle?.let { println(it.asPreviewLine()) }
var confirmationCount = 0
confirmDraft {
    confirmationCount += 1
    println("Confirmed: ${configuredConfirmationDraft.title}")
}
println("Original title: ${originalConfirmationDraft.title}")
println("Ready: ${configuredConfirmationDraft.ready}")
println("Confirmations: $confirmationCount")
```

`apply` configures its receiver and returns that same object. Both draft variables therefore observe the new title; this operation does not create the copy from Lesson 3. Inside `apply`, unqualified `title` and `ready` refer to the draft properties.

The safe-call guard affects only the preview block. `confirmDraft` runs regardless of whether a preview title exists. Defining or passing the lambda alone is not what prints the confirmation: `onConfirmed()` invokes it. The count is one because the function calls it once.

For the null-selected-title test, replace only its initializer with null. Expect:

```text
Confirmed: Pack bag
Original title: Pack bag
Ready: true
Confirmations: 1
```

For the empty-selected-title test, replace only its initializer with `""`. Expect:

```text
Preview: []
Confirmed: Pack bag
Original title: Pack bag
Ready: true
Confirmations: 1
```

A null value skips `let`; an empty String enters it. Neither test changes the configured draft. The confirmation is a printed simulation, not proof of persistent storage.

</details>

### Explain the behavior you observed

Use your test output to explain why the original draft title changed after `apply`. Then explain why confirmation still occurs with a null selected title, which line invokes the callback, and how your output shows the callback ran once. Contrast the null and empty preview outputs.

In [ ]:
Your explanation after testing:
Original draft title after apply:
Why confirmation runs with a null selected title:
Callback invocation line and count evidence:
Null versus empty preview output:


<details>
<summary>Show answer</summary>

`Original title: Pack bag` is consistent with `apply` updating and returning the existing draft object. The callback call is outside the guarded `let`, so a null selection skips only the preview. Inside `confirmDraft`, `onConfirmed()` invokes the action; `Confirmations: 1` records one invocation. Null produces no preview line. An empty string produces `Preview: []`, because it is still a non-null value.

</details>

## Summary

- `() -> Unit` describes an action with no parameters and no meaningful result. Passing an action and invoking it are separate steps.
- A callback runs when the receiving code invokes it. Our simulation invokes it immediately.
- An extension uses receiver syntax without changing the receiver's class. Inside the extension, `this` refers to that receiver.
- `let` provides an argument and returns the block result. `?.let` skips null, but an empty string still enters the block.
- `apply` configures and returns the same receiver object; it does not copy it.
- `object` declares a shared object. A companion object belongs inside a class and can expose members through the class name.
- An eligible `lateinit var` must be assigned before its first read.

Next, we will combine earlier Kotlin skills into a small in-memory notes model. There is no new syntax in that final lesson; the goal is to make existing pieces work together.

## Reflection

Read these declarations:

```kotlin
object PracticeLabels {
    val ready = "Ready"
}
class PracticeDraftDefaults {
    companion object {
        fun defaultTitle(): String = "New reminder"
    }
}
```

Identify which declaration creates a named shared object and which places a companion object inside a class. Write how you would read the ready label and call the default-title function without constructing a `PracticeDraftDefaults` instance. Finally, describe one reason a real app might accept a callback, and state why this lesson's callback simulation does not establish asynchronous behavior.

In [ ]:
Your reflection:
Named shared object:
Companion inside a class:
Ready-label access and default-title call:
Possible callback use in an app:
Why this simulation is not asynchronous:


<details>
<summary>Show answer</summary>

`PracticeLabels` is the named shared object; read its property with `PracticeLabels.ready`. `PracticeDraftDefaults` contains the companion; call `PracticeDraftDefaults.defaultTitle()` through the class name. No `PracticeDraftDefaults()` instance is needed for that call. A real app might accept an action to run when a user confirms an edit. In this lesson, the callback is invoked immediately inside an ordinary function call; no later event, background task, or Android event system is implemented.

</details>

## Supplemental Reading

- [Kotlin function types and lambdas](https://kotlinlang.org/docs/lambdas.html) — function parameters, return types, and passing behavior.
- [Kotlin extensions](https://kotlinlang.org/docs/extensions.html) — receiver syntax and extension behavior.
- [Kotlin scope functions](https://kotlinlang.org/docs/scope-functions.html) — the receiver and result rules of `let` and `apply`.
- [Kotlin object declarations](https://kotlinlang.org/docs/object-declarations.html) — named objects and companion objects.
- [Kotlin properties](https://kotlinlang.org/docs/properties.html) — property initialization, including `lateinit` restrictions.